#### Vector Search with minsearch

In [1]:
# Load the data as an array of object
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
# Convert the data as an array of string
texts = [doc["question"]+" "+doc["answer"] for doc in documents]

In [8]:
# Chose the model for embedding. turn each string into a vectores
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2") 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
from tqdm.auto import tqdm

In [21]:
# Convert the into vector in batch
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [25]:
vectors[0].shape

(384,)

In [26]:
# We turn vectors into a 2-dimensional array (matrix) where
import numpy as np
X = np.array(vectors)

In [27]:
X.shape

(1350, 384)

In [ ]:
# Creating the index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [30]:
# Searching
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [31]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [32]:
# Filtering by course
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)